# Geolocation Transformer

A country-level geolocator trained on Google Street View images. Given 4 panoramic images (N/E/S/W) of a location, the model predicts which country it is in.

## Architecture

| Component | Detail |
|---|---|
| Backbone | CLIP ViT-L/14-336 — **frozen**, used as a feature extractor |
| Head | Single linear layer (768 → N countries) |
| Input | 4 street view images per location, CLIP-embedded and averaged into one 768-dim vector |
| Loss | α × geographic loss + (1−α) × cross-entropy × 100 |

The **geographic loss** penalizes predictions by √(geodesic distance) to the actual country centroid, so predicting a nearby country is penalized less than predicting one on the opposite side of the world.

## Results

| Metric | Value |
|---|---|
| Top-1 Accuracy | — |
| Top-5 Accuracy | — |
| Median km Error | — |

*Update after training. [PIGEON](https://arxiv.org/abs/2307.05845) reports a median error of 40.5 km at the image level for comparison.*

## Data

- ~50,000 street view images across 12,500 locations (4 headings each)
- Countries weighted by `pop^0.30 + area^0.25 + temperature=350` to balance large and small countries
- Raw images stored in S3 (`geolocation-transformer-data`), CLIP embeddings cached locally to avoid re-running the backbone every run

## References

- [PIGEON: Predicting Image Geolocations](https://arxiv.org/abs/2307.05845) — Haas et al., 2023
- [CLIP: Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020) — Radford et al., 2021


In [2]:
from google.colab import drive

drive.mount('/content/drive/')
DRIVE_PATH = '/content/drive/MyDrive/Geolocation'

Mounted at /content/drive/


In [5]:
# ── SETUP: Run once to connect to GitHub, then skip ──────────────────
# Only run this cell when you need to push to GitHub

!git config --global user.email "nicnad7788@gmail.com"
!git config --global user.name "thenicsusanto"
!git clone https://github.com/thenicsusanto/geolocation-model.git /content/geolocation-model

from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git -C /content/geolocation-model remote set-url origin https://{token}@github.com/thenicsusanto/geolocation-model.git

Cloning into '/content/geolocation-model'...
fatal: could not read Username for 'https://github.com': No such device or address
fatal: cannot change to '/content/geolocation-model': No such file or directory


In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/Geolocation/training.ipynb',
            '/content/geolocation-model/notebooks/training.ipynb')
# copy other files...

!git -C /content/geolocation-model add .
!git -C /content/geolocation-model status
!git -C /content/geolocation-model commit -m "Uploading geolocation transformer files and project"
!git -C /content/geolocation-model push origin main

In [ ]:
from transformers import CLIPProcessor, CLIPModel

model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14-336")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14-336")

In [ ]:
from PIL import Image
import requests
import torch
import os
import pandas as pd
import numpy as np
import math
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from geopy.distance import geodesic
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

if torch.cuda.is_available():
  device = 'cuda'
else:
  device = 'cpu'

print(device)

In [ ]:
!pip install boto3 webdataset -q

In [ ]:
df_classes = pd.read_csv(os.path.join(DRIVE_PATH, 'Supported Countries.csv'))
df_metadata = pd.read_csv(os.path.join(DRIVE_PATH, 'metadata.csv'))
classes = df_classes.columns.tolist()
print(classes)

In [ ]:
# country-coord.csv pulled from https://gist.github.com/metal3d/5b925077e66194551df949de64e910f6
print(classes) # countries we want to support
df = pd.read_csv(os.path.join(DRIVE_PATH, 'country-coord.csv')) # approximate centers of each country (to get a rough idea of distance between them)
df.head()

filtered_df = df[df['Country'].isin(classes)]  # filter country coords by countries we want to support

n = len(filtered_df)
distance_matrix = np.zeros((n, n))

# calculate metric over each pair of countries
for i in range(n):
    for j in range(n):
        if i == j:
            distance_matrix[i][j] = 0
        else:
            dist = geodesic(
                (filtered_df.iloc[i]['Latitude (average)'], filtered_df.iloc[i]['Longitude (average)']),
                (filtered_df.iloc[j]['Latitude (average)'], filtered_df.iloc[j]['Longitude (average)'])
            ).kilometers
            distance_matrix[i][j] = math.sqrt(dist) # take square root normalize, this is an educated guess to what will help us make loss lower

dist_df = pd.DataFrame(distance_matrix, index=filtered_df['Country'], columns=filtered_df['Country'])
dist_df = dist_df.reindex(index=classes, columns=classes)

In [ ]:
dist_matrix_tensor = torch.from_numpy(dist_df.values).float().to(device)
print(filtered_df)

def calculate_geographic_loss(logits, target_label, distance_matrix):
  probs = torch.softmax(logits, dim=-1)

  distances_from_actual = distance_matrix[target_label] # list of how far away every country is from country at target_label
  expected_distance = torch.sum(probs * distances_from_actual, dim=-1) # array of weighted guesses with distances
  #print(f"EXPECTED DISTANCE FROM LOSS: {expected_distance}")

  return expected_distance.mean()

In [ ]:
ce_loss_fn = nn.CrossEntropyLoss()

def combined_loss(logits, target_idx, distance_matrix, alpha=0.5):
    geo_loss = calculate_geographic_loss(logits, target_idx, distance_matrix)
    ce_loss = ce_loss_fn(logits, target_idx)
    return (alpha * geo_loss) + ((1 - alpha) * ce_loss * 100)  # scale CE to similar magnitude

In [ ]:
# After creating dist_df, reorder to match classes
dist_df = dist_df.reindex(index=classes, columns=classes)
dist_matrix_tensor = torch.from_numpy(dist_df.values).float().to(device)
print("Classes order:", classes)
print("Dist matrix order:", dist_df.index.tolist())
print("Match:", classes == dist_df.index.tolist())

# Full Training Loop

In [ ]:
for param in model.parameters():
    param.requires_grad = False
model.to(device)
model.eval()

In [ ]:
import boto3
import webdataset as wds
import json
import os
from google.colab import userdata

BUCKET = 'geolocation-transformer-data'
REGION = 'us-west-2'

os.environ['AWS_ACCESS_KEY_ID']     = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION']    = REGION

session = boto3.Session(
    aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name=REGION,
)
s3_client = session.client('s3')

embeddings_cache_path = os.path.join(DRIVE_PATH, 'embeddings_cache.pt')

# Step 1: Load existing cache (dict keyed by location_id)
if os.path.exists(embeddings_cache_path):
    data = torch.load(embeddings_cache_path)
    if 'cache' in data:
        embeddings_cache = data['cache']
        print(f"Loaded {len(embeddings_cache)} existing embeddings from cache")
    else:
        print("Old format detected — starting fresh from S3 shards.")
        embeddings_cache = {}
else:
    embeddings_cache = {}
    print("No cache found, will compute all embeddings from S3 shards")

# Step 2: Discover shards in S3
response   = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='shards/')
shard_keys = sorted([obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.tar')])
print(f"Found {len(shard_keys)} shards in S3")

shards = [f"pipe:aws s3 cp s3://{BUCKET}/{key} -" for key in shard_keys]

# Step 3: Stream shards and compute only missing embeddings in batches
BATCH_SIZE = 16  # 16 locations = 64 images per GPU call

if shards:
    dataset = (
        wds.WebDataset(shards, shardshuffle=False)
        .decode("pil")
        .to_tuple("h0.jpg", "h90.jpg", "h180.jpg", "h270.jpg", "json")
    )

    new_count = 0
    batch_images, batch_meta = [], []

    def process_batch(batch_images, batch_meta):
        n = len(batch_images)
        all_pixels = processor(
            images=[img for loc in batch_images for img in loc], return_tensors="pt"
        )['pixel_values'].to(device)

        with torch.no_grad():
            vision_output = model.get_image_features(pixel_values=all_pixels)
            embeds = vision_output.pooler_output.view(n, 4, -1).mean(dim=1)

        for i, m in enumerate(batch_meta):
            embeddings_cache[m['location_id']] = {
                'embedding': embeds[i].cpu(),
                'country':   m['country']
            }
        return n

    for h0, h90, h180, h270, meta in tqdm(dataset, desc="Computing embeddings"):
        if meta['location_id'] in embeddings_cache or meta['country'] not in classes:
            continue

        batch_images.append([h0, h90, h180, h270])
        batch_meta.append(meta)

        if len(batch_images) < BATCH_SIZE:
            continue

        new_count += process_batch(batch_images, batch_meta)
        batch_images, batch_meta = [], []

    if batch_images:
        new_count += process_batch(batch_images, batch_meta)

    if new_count > 0:
        torch.save({'cache': embeddings_cache}, embeddings_cache_path)
        print(f"Added {new_count} new embeddings. Total: {len(embeddings_cache)}. Saved to {embeddings_cache_path}")
    else:
        print(f"Cache already up to date — {len(embeddings_cache)} embeddings, nothing new to compute")
else:
    print("No shards found in S3 — check that sharding has completed")

# Step 4: Build training tensors from cache
valid_entries = [
    (v['embedding'], classes.index(v['country']))
    for v in embeddings_cache.values()
    if v['country'] in classes
]

all_embeddings = torch.stack([e for e, _ in valid_entries])
all_targets    = torch.tensor([t for _, t in valid_entries])
print(f"Training tensors ready — embeddings: {all_embeddings.shape}, targets: {all_targets.shape}")


In [ ]:
class EmbeddingDataset(Dataset):
    def __init__(self, embeddings, targets):
        self.embeddings = embeddings
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return self.embeddings[idx], self.targets[idx]

# Build ordered location ID list matching all_embeddings / all_targets
all_location_ids = [loc_id for loc_id, v in embeddings_cache.items() if v['country'] in classes]

# Split 80% training 20% val
train_idx, val_idx = train_test_split(range(len(all_targets)), test_size=0.2, random_state=42)
val_location_ids = [all_location_ids[i] for i in val_idx]

train_dataset = EmbeddingDataset(all_embeddings[train_idx], all_targets[train_idx])
val_dataset = EmbeddingDataset(all_embeddings[val_idx], all_targets[val_idx])

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")


In [ ]:
!pip install wandb -q
import wandb

wandb.login(key=userdata.get('WANDB_API_KEY')) # uncomment to skip prompt

ALPHA      = 0.5
LR         = 1e-3
NUM_EPOCHS = 15

wandb.init(
    project="geolocation-transformer",
    config={
        "alpha":        ALPHA,
        "lr":           LR,
        "num_epochs":   NUM_EPOCHS,
        "num_classes":  len(classes),
        "architecture": "frozen CLIP ViT-L/14-336 + linear head",
    }
)


In [ ]:
classifier = nn.Linear(768, len(classes)).to(device)
optimizer = torch.optim.Adam(classifier.parameters(), lr=LR)

# Lookups for median km metric
loc_to_coords = df_metadata.set_index('location_id')[['lat', 'lng']].to_dict('index')
val_actual_coords = [
    (loc_to_coords[all_location_ids[i]]['lat'], loc_to_coords[all_location_ids[i]]['lng'])
    if all_location_ids[i] in loc_to_coords else None
    for i in val_idx
]

country_centroids = {}
for idx, country in enumerate(classes):
    match = filtered_df[filtered_df['Country'] == country]
    if len(match) > 0:
        country_centroids[idx] = (
            match.iloc[0]['Latitude (average)'],
            match.iloc[0]['Longitude (average)']
        )

PATIENCE        = 3
best_val_loss   = float('inf')
epochs_no_improve = 0
best_ckpt_path  = os.path.join(DRIVE_PATH, 'classifier_best.pt')

for epoch in range(NUM_EPOCHS):
    # Train
    classifier.train()
    total_train_loss = 0
    total_train_geo  = 0
    total_train_ce   = 0

    for embeds, targets in train_loader:
        embeds, targets = embeds.to(device), targets.to(device)

        logits   = classifier(embeds)
        geo_loss = calculate_geographic_loss(logits, targets, dist_matrix_tensor)
        ce_loss  = ce_loss_fn(logits, targets)
        loss     = (ALPHA * geo_loss) + ((1 - ALPHA) * ce_loss * 100)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        total_train_geo  += geo_loss.item()
        total_train_ce   += ce_loss.item()

    # Validation
    classifier.eval()
    total_val_loss = 0
    total_val_geo  = 0
    total_val_ce   = 0
    val_preds      = []

    with torch.no_grad():
        for embeds, targets in val_loader:
            embeds, targets = embeds.to(device), targets.to(device)

            logits   = classifier(embeds)
            geo_loss = calculate_geographic_loss(logits, targets, dist_matrix_tensor)
            ce_loss  = ce_loss_fn(logits, targets)
            loss     = (ALPHA * geo_loss) + ((1 - ALPHA) * ce_loss * 100)

            total_val_loss += loss.item()
            total_val_geo  += geo_loss.item()
            total_val_ce   += ce_loss.item()
            val_preds.extend(logits.argmax(dim=-1).cpu().tolist())

    # Median km error
    distances = [
        geodesic(country_centroids[pred], actual).kilometers
        for pred, actual in zip(val_preds, val_actual_coords)
        if actual is not None and pred in country_centroids
    ]
    median_km = np.median(distances) if distances else float('nan')

    n_train = len(train_loader)
    n_val   = len(val_loader)
    avg_val = total_val_loss / n_val

    wandb.log({
        "epoch":          epoch + 1,
        "train/loss":     total_train_loss / n_train,
        "train/geo_loss": total_train_geo  / n_train,
        "train/ce_loss":  total_train_ce   / n_train,
        "val/loss":       avg_val,
        "val/geo_loss":   total_val_geo    / n_val,
        "val/ce_loss":    total_val_ce     / n_val,
        "val/median_km":  median_km,
    })

    improved = avg_val < best_val_loss
    if improved:
        best_val_loss = avg_val
        epochs_no_improve = 0
        torch.save(classifier.state_dict(), best_ckpt_path)
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train: {total_train_loss/n_train:.4f} (geo={total_train_geo/n_train:.4f}, ce={total_train_ce/n_train:.4f}) | "
          f"Val: {avg_val:.4f} | Median km: {median_km:.1f}"
          + (" ✓ saved" if improved else f" (no improvement {epochs_no_improve}/{PATIENCE})"))

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1} — val loss did not improve for {PATIENCE} epochs.")
        break

wandb.finish()
print(f"Best checkpoint saved to {best_ckpt_path} (val loss: {best_val_loss:.4f})")


In [ ]:
SWEEP_EPOCHS = 7  # fewer epochs to keep the sweep fast

sweep_config = {
    "method": "grid",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "alpha": {"values": [0.1, 0.3, 0.5, 0.7, 0.9]},
    },
}

def train_sweep():
    with wandb.init() as run:
        alpha = wandb.config.alpha

        sweep_classifier = nn.Linear(768, len(classes)).to(device)
        sweep_optimizer   = torch.optim.Adam(sweep_classifier.parameters(), lr=LR)

        for epoch in range(SWEEP_EPOCHS):
            sweep_classifier.train()
            total_train_loss = 0
            for embeds, targets in train_loader:
                embeds, targets = embeds.to(device), targets.to(device)
                logits   = sweep_classifier(embeds)
                geo_loss = calculate_geographic_loss(logits, targets, dist_matrix_tensor)
                ce_loss  = ce_loss_fn(logits, targets)
                loss     = (alpha * geo_loss) + ((1 - alpha) * ce_loss * 100)
                sweep_optimizer.zero_grad()
                loss.backward()
                sweep_optimizer.step()
                total_train_loss += loss.item()

            sweep_classifier.eval()
            total_val_loss = 0
            val_preds = []
            with torch.no_grad():
                for embeds, targets in val_loader:
                    embeds, targets = embeds.to(device), targets.to(device)
                    logits   = sweep_classifier(embeds)
                    geo_loss = calculate_geographic_loss(logits, targets, dist_matrix_tensor)
                    ce_loss  = ce_loss_fn(logits, targets)
                    loss     = (alpha * geo_loss) + ((1 - alpha) * ce_loss * 100)
                    total_val_loss += loss.item()
                    val_preds.extend(logits.argmax(dim=-1).cpu().tolist())

            distances = [
                geodesic(country_centroids[pred], actual).kilometers
                for pred, actual in zip(val_preds, val_actual_coords)
                if actual is not None and pred in country_centroids
            ]
            median_km = np.median(distances) if distances else float('nan')

            wandb.log({
                "epoch":         epoch + 1,
                "alpha":         alpha,
                "train/loss":    total_train_loss / len(train_loader),
                "val/loss":      total_val_loss   / len(val_loader),
                "val/median_km": median_km,
            })

sweep_id = wandb.sweep(sweep_config, project="geolocation-transformer")
wandb.agent(sweep_id, function=train_sweep)


In [ ]:
def predict_country(location_id, image_dir, model, classifier, processor, classes, top_k=5):
    """Predict country for a given location ID."""

    images = []
    for h in [0, 90, 180, 270]:
        img_path = os.path.join(image_dir, f"{location_id}_h{h}.jpg")
        images.append(Image.open(img_path).convert("RGB"))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    directions = ['North', 'East', 'South', 'West']
    for ax, img, direction in zip(axes, images, directions):
        ax.imshow(img)
        ax.set_title(direction)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

    pixel_values = processor(images=images, return_tensors="pt")['pixel_values'].to(device)

    classifier.eval()
    with torch.no_grad():
        vision_output = model.get_image_features(pixel_values=pixel_values)
        image_embeds = vision_output.pooler_output.view(1, 4, -1).mean(dim=1)
        logits = classifier(image_embeds)
        probs = torch.softmax(logits, dim=-1)

    top_probs, top_indices = probs.topk(top_k)
    print(f"Top {top_k} predictions:")
    for prob, idx in zip(top_probs[0], top_indices[0]):
        print(f"  {classes[idx]}: {prob.item():.2%}")

    return classes[top_indices[0][0].item()]


# Test on a random sample from the actual validation set
val_metadata = df_metadata[df_metadata['location_id'].isin(val_location_ids)]
test_row = val_metadata.sample(1).iloc[0]

test_id = test_row['location_id']
actual_country = test_row['country']

print(f"Location: {test_id}")
print(f"Actual country: {actual_country}\n")

classifier = nn.Linear(768, len(classes)).to(device)
classifier.load_state_dict(torch.load(os.path.join(DRIVE_PATH, 'classifier_best.pt'), map_location=device))
classifier.eval()

model.to(device)  # ensure CLIP is on the correct device before inference

predicted = predict_country(
    test_id,
    os.path.join(DRIVE_PATH, 'images'),
    model,
    classifier,
    processor,
    classes
)

print(f"\nPredicted: {predicted}")
print(f"Correct: {predicted == actual_country}")


In [ ]:
!pip install seaborn -q

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Load the trained classifier
classifier = nn.Linear(768, len(classes)).to(device)
classifier.load_state_dict(torch.load(os.path.join(DRIVE_PATH, 'classifier.pt')))
classifier.eval()

# Run on full val set using cached embeddings (fast — no image loading)
top1_correct = 0
top5_correct = 0
all_actual = []
all_predicted = []

with torch.no_grad():
    for idx in tqdm(val_idx, desc="Evaluating"):
        embeds = all_embeddings[idx].unsqueeze(0).to(device)
        target = all_targets[idx].item()

        logits = classifier(embeds)
        probs = torch.softmax(logits, dim=-1)

        top1_pred = probs.argmax(dim=-1).item()
        top5_preds = probs.topk(5).indices[0].tolist()

        if top1_pred == target:
            top1_correct += 1
        if target in top5_preds:
            top5_correct += 1

        all_actual.append(target)
        all_predicted.append(top1_pred)

n = len(val_idx)
print(f"{'='*35}")
print(f"FULL VALIDATION SET RESULTS ({n} samples)")
print(f"Top-1 Accuracy: {top1_correct/n*100:.2f}%")
print(f"Top-5 Accuracy: {top5_correct/n*100:.2f}%")
print(f"{'='*35}")

# Force cm to cover all class indices so indexing with len(classes) is safe
cm = confusion_matrix(all_actual, all_predicted, labels=list(range(len(classes))))

# Top confused pairs
errors = []
for i in range(len(classes)):
    for j in range(len(classes)):
        if i != j and cm[i][j] > 0:
            errors.append((cm[i][j], classes[i], classes[j]))

errors.sort(reverse=True)
print(f"\nTop 15 confused pairs (actual → predicted):")
for count, actual, predicted in errors[:15]:
    print(f"  {actual} → {predicted}: {count}x")

# Confusion matrix heatmap (only classes that appear in val set)
active_indices = sorted(set(all_actual + all_predicted))
active_classes = [classes[i] for i in active_indices]
cm_filtered = cm[np.ix_(active_indices, active_indices)]

plt.figure(figsize=(20, 16))
sns.heatmap(
    cm_filtered,
    xticklabels=active_classes,
    yticklabels=active_classes,
    cmap='Blues',
    fmt='d',
    annot=len(active_classes) <= 30,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (Validation Set)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Per-country accuracy W&B table
per_country_rows = []
for i, country in enumerate(classes):
    total = cm[i].sum()
    if total == 0:
        continue
    correct = cm[i][i]
    per_country_rows.append({
        "country":       country,
        "top1_accuracy": round(correct / total * 100, 2),
        "correct":       int(correct),
        "total":         int(total),
    })

per_country_rows.sort(key=lambda x: x["top1_accuracy"])

wandb.init(project="geolocation-transformer", job_type="eval")
wandb.log({
    "eval/top1_accuracy": top1_correct / n * 100,
    "eval/top5_accuracy": top5_correct / n * 100,
    "eval/per_country_accuracy": wandb.Table(
        columns=["country", "top1_accuracy", "correct", "total"],
        data=[[r["country"], r["top1_accuracy"], r["correct"], r["total"]] for r in per_country_rows]
    ),
})
wandb.finish()
print("\nPer-country accuracy logged to W&B.")


In [ ]:
!pip install geopandas -q

import geopandas as gpd

world = gpd.read_file(
    "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson"
)

# Centroid lookup: country name -> (lng, lat) for plotting
centroid_lookup = {
    row['Country']: (row['Longitude (average)'], row['Latitude (average)'])
    for _, row in filtered_df.iterrows()
}

TOP_N = 20
plottable_errors = [
    (count, actual, predicted)
    for count, actual, predicted in errors
    if actual in centroid_lookup and predicted in centroid_lookup
][:TOP_N]

max_count = plottable_errors[0][0] if plottable_errors else 1

fig, ax = plt.subplots(figsize=(22, 11))
world.plot(ax=ax, color='#e8e8e8', edgecolor='white', linewidth=0.5)

for count, actual, predicted in plottable_errors:
    lng1, lat1 = centroid_lookup[actual]
    lng2, lat2 = centroid_lookup[predicted]

    weight = count / max_count
    ax.plot(
        [lng1, lng2], [lat1, lat2],
        color='crimson', alpha=0.3 + 0.6 * weight,
        linewidth=0.5 + 4.0 * weight, solid_capstyle='round', zorder=3
    )
    ax.scatter([lng1, lng2], [lat1, lat2],
               color='crimson', s=15, zorder=4, alpha=0.6 + 0.4 * weight)

ax.set_title(
    f'Top {len(plottable_errors)} Most Confused Country Pairs (Validation Set)\n'
    'Line weight = confusion count',
    fontsize=14
)
ax.set_axis_off()
plt.tight_layout()

save_path = os.path.join(DRIVE_PATH, 'confusion_map.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved confusion_map.png to Drive")
